In [1]:
from pymol import cmd
from IPython.display import Image, display
from pymol import util
import math
from pyrosetta import *
from utils import perform_mutation, relax_structure
import random
import tempfile
from pyrosetta.rosetta.protocols.analysis import InterfaceAnalyzerMover

/home/jcape/coding/Toxin-Engineering/utils.py:10: UserWarning: Import of 'rosetta' as a top-level module is deprecated and may be removed in 2018, import via 'pyrosetta.rosetta'.
  from rosetta.core.pack.task import TaskFactory


┌───────────────────────────────────────────────────────────────────────────────┐
│                                  PyRosetta-4                                  │
│               Created in JHU by Sergey Lyskov and PyRosetta Team              │
│               (C) Copyright Rosetta Commons Member Institutions               │
│                                                                               │
│ NOTE: USE OF PyRosetta FOR COMMERCIAL PURPOSES REQUIRES PURCHASE OF A LICENSE │
│          See LICENSE.PyRosetta.md or email license@uw.edu for details         │
└───────────────────────────────────────────────────────────────────────────────┘
PyRosetta-4 2026 [Rosetta PyRosetta4.conda.ubuntu.cxx11thread.serialization.Ubuntu.python313.Release 2026.26+release.f8a8eab4344af49c25e1e5db84ce25fec05eea27 2026-06-25T18:00:21] retrieved from: http://www.pyrosetta.org



ERROR: Cannot open file "coding/Toxin-Engineering/pdb_files/7K18_relax.pdb"
ERROR:: Exit from: /home/benchmark/rosetta/source/src/core/import_pose/import_pose.cc line: 387


RuntimeError: 

File: /home/benchmark/rosetta/source/src/core/import_pose/import_pose.cc:387
[ ERROR ] UtilityExitException
ERROR: Cannot open file "coding/Toxin-Engineering/pdb_files/7K18_relax.pdb"



In [ ]:
def distance_finder(pdb):
    # Reset cell to ensure no other proteins are in the current session
    cmd.delete("all")
    
    # perform basic tasks
    cmd.load(pdb) # load pdb file
    # cmd.iterate("resi 43", "print(chain, resi, resn, name)")
    # select histdines
    his = cmd.get_atom_coords('chain B and resi 43 and name ND1')
    his2 = cmd.get_atom_coords('chain B and resi 15 and name NE2')


    resn = cmd.get_model("chain A and resi 1612").atom[0].resn

    if resn == "SER":
        ser = cmd.get_atom_coords("chain A and resi 1612 and name OG")
        distance = math.dist(ser, his)
        distance2 = math.dist(ser, his2)

    elif resn == "ASP":
        asp1 = cmd.get_atom_coords("chain A and resi 1612 and name OD1")
        asp2 = cmd.get_atom_coords("chain A and resi 1612 and name OD2")
        d1 = math.dist(asp1, his)
        d2 = math.dist(asp2, his)
        distance = min(d1, d2)

        d1_2 = math.dist(asp1, his2)
        d2_2 = math.dist(asp2, his2)
        distance2 = min(d2_2, d1_2)


    return distance, distance2

In [3]:
def energy_finder(pdb):
    init()

    scorefxn = get_score_function()
    pose = pose_from_pdb(pdb)

    energy = scorefxn(pose)
    return energy

In [2]:
def delta_g(pdb):
    pose = pose_from_pdb(pdb)

    scorefxn = get_fa_scorefxn()
    iam = InterfaceAnalyzerMover(
    1,          # interface jump
    False,      # tracer output
    scorefxn)
    iam.apply(pose)
    return iam.get_interface_dG()

In [11]:
def scoring_function(pose_or_pdb, weight1=2, weight2=1):

    if isinstance(pose_or_pdb, str):
        pdb = pose_or_pdb

    else:
        with tempfile.NamedTemporaryFile(suffix=".pdb") as tmp:
            pose_or_pdb.dump_pdb(tmp.name)

            

            return delta_g(tmp.name)


    return delta_g(pdb)

In [12]:
scoring_function('pdb_files/7K18_relax.pdb')

core.import_pose.import_pose: File 'pdb_files/7K18_relax.pdb' automatically determined to be of type PDB from contents.
core.conformation.Conformation: [ WARNING ] missing heavyatom:  OXT on residue ALA:CtermProteinFull 100
core.conformation.Conformation: Found disulfide between residues 112 165
core.conformation.Conformation: Found disulfide between residues 116 137
core.conformation.Conformation: Found disulfide between residues 123 147
core.conformation.Conformation: Found disulfide between residues 127 149
core.scoring.ScoreFunctionFactory: SCOREFUNCTION: ref2015
protocols.analysis.InterfaceAnalyzerMover: Using normal constructor
protocols.evaluation.ChiWellRmsdEvaluatorCreator: Evaluation Creator active ...
protocols.analysis.InterfaceAnalyzerMover: Making an interface set with default calculator.
protocols.analysis.InterfaceAnalyzerMover: Detecting disulfides in the separated pose.
core.conformation.Conformation: Found disulfide between residues 112 165
core.conformation.Conforma

-35.63297887804714

In [6]:
pose = pose_from_pdb('pdb_files/7K18_relax_59M_61S_62S_63P.pdb')

core.import_pose.import_pose: File 'pdb_files/7K18_relax_59M_61S_62S_63P.pdb' automatically determined to be of type PDB from contents.
core.conformation.Conformation: Found disulfide between residues 112 165
core.conformation.Conformation: Found disulfide between residues 116 137
core.conformation.Conformation: Found disulfide between residues 123 147
core.conformation.Conformation: Found disulfide between residues 127 149


In [13]:
def random_mutation(pdb_or_pose, start = 130, end = 150):
    amino_acids = ["A","R","N","D","C","Q","E","G","H","I",
                   "L","K","M","F","P","S","T","W","Y","V"]
    aa = random.choice(amino_acids)
    pos = random.choice([i for i in range(start, end) if i != 143])

    if isinstance(pdb_or_pose, str):
        return perform_mutation(pdb_or_pose, pos, aa)

    else:
        with tempfile.NamedTemporaryFile(suffix=".pdb") as tmp:
            pdb_or_pose.dump_pdb(tmp.name)
            return perform_mutation(tmp.name, pos, aa)


In [15]:
pose = random_mutation('pdb_files/7K18_relax_59M_61S_62S_63P.pdb')
scoring_function(pose)

core.import_pose.import_pose: File 'pdb_files/7K18_relax_59M_61S_62S_63P.pdb' automatically determined to be of type PDB from contents.
core.conformation.Conformation: Found disulfide between residues 112 165
core.conformation.Conformation: Found disulfide between residues 116 137
core.conformation.Conformation: Found disulfide between residues 123 147
core.conformation.Conformation: Found disulfide between residues 127 149
core.scoring.ScoreFunctionFactory: SCOREFUNCTION: ref2015
core.pack.task: Packer task: initialize from command line()
core.select.residue_selector.NeighborhoodResidueSelector: [ WARNING ] ################ Cloning pose and building neighbor graph ################
core.select.residue_selector.NeighborhoodResidueSelector: [ WARNING ] Ensure that pose is either scored or has update_residue_neighbors() called
core.select.residue_selector.NeighborhoodResidueSelector: [ WARNING ] before using NeighborhoodResidueSelector for maximum performance!
core.select.residue_selector

-30.25864572141345

In [16]:
def simulation(pdb, temp=5, cooling_rate=0.95, iterations=1000):

    current_pose = pose_from_pdb(pdb)

    current_score = scoring_function(current_pose, 5, 2)

    best_pose = current_pose
    best_score = current_score

    for i in range(iterations):
        print('iteration' + str(i))

        new_pose = random_mutation(current_pose, )

        mutant_score = scoring_function(new_pose, 5, 2)

        delta = mutant_score - current_score

        if delta < 0:
            current_pose = new_pose
            current_score = mutant_score

        else:
            prob = math.exp(-delta / temp)

            if random.random() < prob:
                current_pose = new_pose
                current_score = mutant_score

        if current_score < best_score:
            best_pose = current_pose
            best_score = current_score

        temp *= cooling_rate

    print("Best score:", best_score)

    return best_pose


In [17]:
best_pose = simulation('pdb_files/7K18_relax_59M_61S_62S_63P.pdb', iterations=100, temp=10)

core.import_pose.import_pose: File 'pdb_files/7K18_relax_59M_61S_62S_63P.pdb' automatically determined to be of type PDB from contents.
core.conformation.Conformation: Found disulfide between residues 112 165
core.conformation.Conformation: Found disulfide between residues 116 137
core.conformation.Conformation: Found disulfide between residues 123 147
core.conformation.Conformation: Found disulfide between residues 127 149
core.import_pose.import_pose: File '/tmp/tmpy39ks8tx.pdb' automatically determined to be of type PDB from contents.
core.conformation.Conformation: Found disulfide between residues 112 165
core.conformation.Conformation: Found disulfide between residues 116 137
core.conformation.Conformation: Found disulfide between residues 123 147
core.conformation.Conformation: Found disulfide between residues 127 149
core.scoring.ScoreFunctionFactory: SCOREFUNCTION: ref2015
protocols.analysis.InterfaceAnalyzerMover: Using normal constructor
protocols.analysis.InterfaceAnalyzerMo

In [18]:
dump_pdb(best_pose, 'experiment_pose.pdb')

True

In [19]:
best_pose.sequence()

'DDQSPEKVNILAKINLLFVAIFTGECIVKMAALRHYYFTNSWNIFDFVVVILSIVGTVMSSSPQKYFFSPTLFRVIRLARIGRILRLIRGAKGIRTLLFAVRDGYIAQPENCVYHCFPGSSGCDTLCKERFSGVKCCMFVTAHGIFCCCNALPDNVGIIVEGEKCHS'

In [ ]:
delta_g('pdb_files/7K18_mutant.pdb')

┌───────────────────────────────────────────────────────────────────────────────┐
│                                  PyRosetta-4                                  │
│               Created in JHU by Sergey Lyskov and PyRosetta Team              │
│               (C) Copyright Rosetta Commons Member Institutions               │
│                                                                               │
│ NOTE: USE OF PyRosetta FOR COMMERCIAL PURPOSES REQUIRES PURCHASE OF A LICENSE │
│          See LICENSE.PyRosetta.md or email license@uw.edu for details         │
└───────────────────────────────────────────────────────────────────────────────┘
PyRosetta-4 2026 [Rosetta PyRosetta4.conda.ubuntu.cxx11thread.serialization.Ubuntu.python313.Release 2026.26+release.f8a8eab4344af49c25e1e5db84ce25fec05eea27 2026-06-25T18:00:21] retrieved from: http://www.pyrosetta.org
core.scoring.ScoreFunctionFactory: SCOREFUNCTION: ref2015
core.import_pose.import_pose: File 'pdb_files/7K18_relax.pdb' auto

-297.20031721381685